In [ ]:
# To fix AttributeError: module 'cv2' has no attribute 'CascadeClassifier'
# Uncomment the line below, run this cell, then restart runtime and run all cells.
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python
!pip install opencv-contrib-python

import os
import cv2
import gc  # Garbage collection tool to forcefully clear RAM cache
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

# =====================================================================
# 1. SETUP PARAMETERS & DIRECTORY PATH VALIDATION
# =====================================================================
# Point this to your extracted image directory inside Google Colab
ZIP_FILE_PATH = "/content/drive/MyDrive/Appblocker/UTKFace.zip"
EXTRACTED_DIR_PATH = "/content/drive/MyDrive/Appblocker/UTKFace_extracted"
CHECKPOINT_DIR = "/content/drive/MyDrive/Appblocker/Model_Checkpoints" # New: Directory for saving model checkpoints
IMG_SIZE = 224
BATCH_SIZE = 32

# Ensure the checkpoint directory exists
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Model checkpoint directory set to: '{CHECKPOINT_DIR}'")

# Check if the zip file exists
if not os.path.exists(ZIP_FILE_PATH):
    raise FileNotFoundError(f"🚨 ERROR: Cannot find zip file at '{ZIP_FILE_PATH}'. Check your sidebar path name.")

# Unzip the file if the extracted directory does not exist
if not os.path.exists(EXTRACTED_DIR_PATH):
    print(f"Unzipping '{ZIP_FILE_PATH}' to '{EXTRACTED_DIR_PATH}'...")
    import zipfile
    with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACTED_DIR_PATH)
    print("Unzipping complete.")
else:
    print(f"Extracted directory '{EXTRACTED_DIR_PATH}' already exists. Skipping unzipping.")

# --- NEW: Adjust EXTRACTED_DIR_PATH to point to the specific 'UTKFace' subdirectory ---
UTKFACE_SUBDIR_PATH = os.path.join(EXTRACTED_DIR_PATH, 'UTKFace')
if os.path.exists(UTKFACE_SUBDIR_PATH):
    EXTRACTED_DIR_PATH = UTKFACE_SUBDIR_PATH
    print(f"Adjusted image source to: '{EXTRACTED_DIR_PATH}' for specific 'UTKFace' content.")
else:
    print(f"Warning: '{UTKFACE_SUBDIR_PATH}' not found. Searching all subdirectories as a fallback.")
# --------------------------------------------------------------------------------------

# Fetch every available image filename from the folder and its subfolders recursively
all_files_full_paths = []
for root, dirs, files in os.walk(EXTRACTED_DIR_PATH):
    for file in files:
        if file.endswith('.jpg'):
            all_files_full_paths.append(os.path.join(root, file))

all_files_full_paths.sort() # Ensure consistent order
print(f"✅ Safe Engine detected {len(all_files_full_paths)} total raw data files.")

if not all_files_full_paths:
    raise ValueError(f"🚨 ERROR: No JPG files found in '{EXTRACTED_DIR_PATH}' or its subdirectories. Please check your dataset.")

# Load OpenCV's pre-trained Face Detection Cascade
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')


# =====================================================================
# 2. RAW TEXT PARSING & STRATEGIC SUBSET SPLITTING
# =====================================================================
file_paths = []
ages = []
age_classes = [] # 0: Kid (<13), 1: Teen (13-19), 2: Adult (>19)

for full_path in all_files_full_paths:
    filename = os.path.basename(full_path) # Extract filename to parse age
    parts = filename.split('_')
    if len(parts) >= 3:
        try:
            age = int(parts[0]) # Corrected to parse the age from the first part
            if age < 13:
                cls = 0
            elif age <= 19:
                cls = 1
            else:
                cls = 2

            ages.append(age)
            age_classes.append(cls)
            file_paths.append(full_path) # Use the full path here
        except ValueError:
            continue

if not file_paths:
    raise ValueError(f"🚨 ERROR: No valid age data could be parsed from the image filenames. Check naming convention.")

file_paths = np.array(file_paths)
ages = np.array(ages, dtype=np.float32)
age_classes = np.array(age_classes, dtype=np.int32)

# Split arrays into 80% Training and 20% Validation pathways
tr_paths, val_paths, tr_ages, val_ages, tr_cls, val_cls = train_test_split(
    file_paths, ages, age_classes, test_size=0.2, random_state=42
)

print(f"📊 Dataset split complete.")
print(f"• Training Files Count  : {len(tr_paths)}")
print(f"• Validation Files Count: {len(val_paths)}")


# =====================================================================
# 3. HIGH-SPEED IN-PLACE IMAGE PROCESSING (RAM CRASH PROTECTION)
# =====================================================================
def preprocess_inplace_image(image_path, out_matrix, index):
    """Processes image and injects it directly into an allocated memory array slot."""
    img = cv2.imread(image_path)
    if img is None:
        out_matrix[index] = 0.0
        return

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5)

    # --- STAGE A: FACE EXTRACTION WITH CENTER CROP FALLBACK ---
    if len(faces) == 0:
        h_raw, w_raw, _ = img.shape
        start_y, end_y = int(h_raw * 0.1), int(h_raw * 0.9)
        start_x, end_x = int(w_raw * 0.1), int(w_raw * 0.9)
        face_roi = img[start_y:end_y, start_x:end_x]
    else:
        x, y, w, h = max(faces, key=lambda b: b[2] * b[3]) # Corrected face area calculation
        face_roi = img[y:y+h, x:x+w]

    # --- STAGE B: ILLUMINATION NORMALIZATION (CLAHE) ---
    ycrcb = cv2.cvtColor(face_roi, cv2.COLOR_BGR2YCrCb)
    channels = list(cv2.split(ycrcb))
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    channels[0] = clahe.apply(channels[0]) # Apply CLAHE to the Y channel
    ycrcb = cv2.merge(channels)
    normalized_rgb = cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2RGB)

    # --- STAGE C: SPATIAL RESIZING ---
    resized_final = cv2.resize(normalized_rgb, (IMG_SIZE, IMG_SIZE))

    # --- STAGE D: NUMERICAL INTENSITY NORMALIZATION ---
    # Direct In-Place assignment to memory slot to prevent Python list RAM inflation
    out_matrix[index] = resized_final.astype(np.float32) / 255.0


# =====================================================================
# 4. BUFFER OVERWRITE MULTI-TASK LAZY GENERATOR
# =====================================================================
class RAMProofLazyGenerator(tf.keras.utils.Sequence):
    """The Keras Data Generator using in-place fixed matrix overwriting."""
    def __init__(self, paths, ages, classes, batch_size=32):
        self.paths = paths
        self.ages = ages
        self.classes = classes
        self.batch_size = batch_size
        # Pre-allocate a single permanent memory slot for one batch of 32 images
        self.batch_images_buffer = np.zeros((batch_size, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)

    def __len__(self):
        return int(np.ceil(len(self.paths) / self.batch_size))

    def __getitem__(self, idx):
        batch_x = self.paths[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_y_age = self.ages[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_y_cls = self.classes[idx * self.batch_size:(idx + 1) * self.batch_size]

        # Reset actual processing length for trailing end-of-dataset odd batches
        current_batch_len = len(batch_x)

        # Overwrite the exact same memory buffer matrix block over and over
        for i in range(current_batch_len):
            preprocess_inplace_image(batch_x[i], self.batch_images_buffer, i)

        # Safe categorical structural transformation mapping
        batch_y_cls_onehot = tf.keras.utils.to_categorical(batch_y_cls, num_classes=3)

        # Forcefully clear memory background leaks actively every 10 steps
        if idx % 10 == 0:
            gc.collect()

        return self.batch_images_buffer[:current_batch_len], {
            "classification_output": batch_y_cls_onehot,
            "regression_output": batch_y_age
        }

# Instantiate our safe streaming objects
train_generator = RAMProofLazyGenerator(tr_paths, tr_ages, tr_cls, batch_size=BATCH_SIZE)
val_generator = RAMProofLazyGenerator(val_paths, val_ages, val_cls, batch_size=BATCH_SIZE)

print("\n🚀 Preprocessing Pipeline Configuration Complete!")
print("The data streams are pre-allocated and ready to run without crashing your 12.7GB RAM.")


Found existing installation: opencv-contrib-python 5.0.0.93
Uninstalling opencv-contrib-python-5.0.0.93:
  Successfully uninstalled opencv-contrib-python-5.0.0.93
  Using cached opencv_contrib_python-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
Using cached opencv_contrib_python-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl (82.1 MB)
Model checkpoint directory set to: '/content/drive/MyDrive/Appblocker/Model_Checkpoints'
Extracted directory '/content/drive/MyDrive/Appblocker/UTKFace_extracted' already exists. Skipping unzipping.
Adjusted image source to: '/content/drive/MyDrive/Appblocker/UTKFace_extracted/UTKFace' for specific 'UTKFace' content.
✅ Safe Engine detected 23708 total raw data files.
📊 Dataset split complete.
• Training Files Count  : 18966
• Validation Files Count: 4742

🚀 Preprocessing Pipeline Configuration Complete!
The data streams are pre-allocated and ready to run without crashing your 12.7GB RAM.


In [ ]:
import tensorflow as tf

# Load Haar Cascade
# =====================================================================
# LOAD HAAR FACE DETECTOR
# =====================================================================

!wget -O /content/haarcascade_frontalface_default.xml \
https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml


cascade_path = "/content/haarcascade_frontalface_default.xml"

face_cascade = cv2.CascadeClassifier(cascade_path)


if face_cascade.empty():
    raise RuntimeError(
        "❌ Haar Cascade failed to load"
    )

print("✅ Haar Cascade loaded successfully")

clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8, 8)
)

def preprocess_inplace_image(image_path, out_matrix, index):
    # print(f"[DEBUG] Processing image: {image_path} (index {index})") # Enable for very verbose logging
    img = cv2.imread(image_path)

    if img is None:
        # print(f"[DEBUG] Skipping unreadable image: {image_path}") # Enable for debug
        out_matrix[index] = np.zeros(
            (IMG_SIZE, IMG_SIZE, 3),
            dtype=np.float32
        )
        return

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5
    )
    # print(f"[DEBUG] Faces detected: {len(faces)} for {os.path.basename(image_path)}") # Enable for debug

    if len(faces) == 0:

        h, w = img.shape[:2]

        sx = int(w * 0.10)
        sy = int(h * 0.10)

        ex = int(w * 0.90)
        ey = int(h * 0.90)

        face = img[sy:ey, sx:ex]

    else:

        x, y, w, h = max(
            faces,
            key=lambda f: f[2] * f[3]
        )

        face = img[y:y+h, x:x+w]
    # print(f"[DEBUG] Face ROI shape: {face.shape} for {os.path.basename(image_path)}") # Enable for debug

    ycrcb = cv2.cvtColor(
        face,
        cv2.COLOR_BGR2YCrCb
    )

    channels = list(cv2.split(ycrcb))

    channels[0] = clahe.apply(channels[0])

    ycrcb = cv2.merge(channels)

    rgb = cv2.cvtColor(
        ycrcb,
        cv2.COLOR_YCrCb2RGB
    )

    rgb = cv2.resize(
        rgb,
        (IMG_SIZE, IMG_SIZE)
    )
    # print(f"[DEBUG] Resized image shape: {rgb.shape} for {os.path.basename(image_path)}") # Enable for debug

    rgb = rgb.astype(np.float32)

    rgb = tf.keras.applications.mobilenet_v3.preprocess_input(rgb)

    out_matrix[index] = rgb


--2026-08-08 04:39:36--  https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 930127 (908K) [text/plain]
Saving to: ‘/content/haarcascade_frontalface_default.xml’

/content/haarcascad 100%[===================>] 908.33K  --.-KB/s    in 0.03s   

2026-08-08 04:39:36 (25.9 MB/s) - ‘/content/haarcascade_frontalface_default.xml’ saved [930127/930127]

✅ Haar Cascade loaded successfully


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# =====================================================================
# 4. IMPROVED RAM-PROOF DATA GENERATOR
# =====================================================================

class RAMProofLazyGenerator(tf.keras.utils.Sequence):
    """
    Efficient Keras data generator that loads images batch-by-batch,
    preprocesses them in-place, and supports shuffling.
    """

    def __init__(
        self,
        paths,
        ages,
        classes,
        batch_size=32,
        shuffle=True
    ):
        self.paths = np.array(paths)
        self.ages = np.array(ages, dtype=np.float32)
        self.classes = np.array(classes, dtype=np.int32)

        self.batch_size = batch_size
        self.shuffle = shuffle

        self.batch_images_buffer = np.zeros(
            (batch_size, IMG_SIZE, IMG_SIZE, 3),
            dtype=np.float32
        )

        self.indices = np.arange(len(self.paths))

        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.paths) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):

        start = idx * self.batch_size
        end = min((idx + 1) * self.batch_size, len(self.paths))

        batch_indices = self.indices[start:end]

        current_batch_size = len(batch_indices)

        for i, sample_index in enumerate(batch_indices):

            preprocess_inplace_image(
                self.paths[sample_index],
                self.batch_images_buffer,
                i
            )

        batch_age = self.ages[batch_indices]

        batch_cls = tf.keras.utils.to_categorical(
            self.classes[batch_indices],
            num_classes=3
        )

        # Removed gc.collect() as it's detrimental to performance and multiprocessing should handle memory better

        return (
            self.batch_images_buffer[:current_batch_size],
            {
                "classification_output": batch_cls,
                "regression_output": batch_age,
            },
        )


In [ ]:
train_generator = RAMProofLazyGenerator(
    tr_paths,
    tr_ages,
    tr_cls,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_generator = RAMProofLazyGenerator(
    val_paths,
    val_ages,
    val_cls,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("\n✅ Data generators created successfully.")
print(f"Training batches   : {len(train_generator)}")
print(f"Validation batches : {len(val_generator)}")


✅ Data generators created successfully.
Training batches   : 593
Validation batches : 149


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# =====================================================================
# 5. IMPROVED MULTI-TASK MODEL
# =====================================================================

print("🏗️ Building MobileNetV3 Multi-Task Network...")

IMG_SIZE = 224

# -----------------------------------------------------------
# Load ImageNet pretrained MobileNetV3-Small
# -----------------------------------------------------------

base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze backbone for Stage-1 training
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# Important for BatchNormalization layers
x = base_model(inputs, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.30)(x)

shared = layers.Dense(
    256,
    activation="relu"
)(x)

shared = layers.BatchNormalization()(shared)

shared = layers.Dropout(0.30)(shared)

# ============================================================
# Classification Head
# ============================================================

cls = layers.Dense(
    128,
    activation="relu"
)(shared)

cls = layers.Dropout(0.20)(cls)

classification_output = layers.Dense(
    3,
    activation="softmax",
    name="classification_output"
)(cls)

# ============================================================
# Regression Head
# ============================================================

reg = layers.Dense(
    128,
    activation="relu"
)(shared)

reg = layers.Dropout(0.20)(reg)

regression_output = layers.Dense(
    1,
    activation="linear",
    name="regression_output"
)(reg)

# -----------------------------------------------------------

model = models.Model(
    inputs=inputs,
    outputs=[
        classification_output,
        regression_output
    ]
)

# -----------------------------------------------------------
# Stage-1 Compile
# -----------------------------------------------------------

model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),

    loss={

        "classification_output":
            tf.keras.losses.CategoricalCrossentropy(),

        "regression_output":
            tf.keras.losses.Huber()

    },

    loss_weights={

        "classification_output":1.0,

        "regression_output":0.05

    },

    metrics={

        "classification_output":["accuracy"],

        "regression_output":["mae"]

    }

)

model.summary()

🏗️ Building MobileNetV3 Multi-Task Network...
4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ MobileNetV3Small    │ (None, 7, 7, 576) │    939,120 │ input_layer_1[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 576)       │          0 │ MobileNetV3Small… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 576)       │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    147,712 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256)       │      1,024 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     32,896 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classification_out… │ (None, 3)         │        387 │ dropout_2[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ regression_output   │ (None, 1)         │        129 │ dropout_3[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,154,164 (4.40 MB)

 Trainable params: 214,532 (838.02 KB)

 Non-trainable params: 939,632 (3.58 MB)

In [ ]:
1e-3

0.001

In [ ]:
1e-5

1e-05

In [ ]:
# =====================================================================
# 6. TWO-STAGE MODEL TRAINING
# =====================================================================

print("\n🚀 Starting Stage-1 Transfer Learning Training...")


from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)


# -----------------------------------------------------------
# Training callbacks
# -----------------------------------------------------------

callbacks = [

    EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    ModelCheckpoint(
        filepath=os.path.join(CHECKPOINT_DIR, "age_model_stage1_epoch_{epoch:02d}.keras"), # Save every epoch to Drive
        monitor="val_loss",
        save_best_only=False, # Changed to False to save every epoch
        save_weights_only=False, # Save the entire model
        verbose=1
    )

]


# -----------------------------------------------------------
# Stage 1: Train only custom heads
# -----------------------------------------------------------

STAGE1_EPOCHS = 5


history_stage1 = model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=STAGE1_EPOCHS,

    callbacks=callbacks

)


print("\n✅ Stage-1 training complete.")



# =====================================================================
# STAGE 2 : Fine tuning MobileNetV3 backbone
# =====================================================================

print("\n🔥 Starting Stage-2 Fine Tuning...")


# Unfreeze MobileNetV3

base_model.trainable = True


# Keep BatchNormalization layers frozen
# This prevents instability with smaller datasets

for layer in base_model.layers:

    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = False



# Recompile with smaller learning rate

model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),

    loss={

        "classification_output":
            tf.keras.losses.CategoricalCrossentropy(),

        "regression_output":
            tf.keras.losses.Huber()

    },

    loss_weights={

        "classification_output":1.0,

        "regression_output":0.05

    },

    metrics={

        "classification_output":
            ["accuracy"],

        "regression_output":
            ["mae"]

    }

)



callbacks_stage2 = [

    EarlyStopping(

        monitor="val_loss",

        patience=5,

        restore_best_weights=True,

        verbose=1

    ),


    ReduceLROnPlateau(

        monitor="val_loss",

        factor=0.2,

        patience=2,

        min_lr=1e-8,

        verbose=1

    ),


    ModelCheckpoint(

        filepath=os.path.join(CHECKPOINT_DIR, "age_model_final_epoch_{epoch:02d}.keras"), # Save every epoch to Drive
        monitor="val_loss",
        save_best_only=False, # Changed to False to save every epoch
        save_weights_only=False, # Save the entire model
        verbose=1

    )

]



STAGE2_EPOCHS = 15


history_stage2 = model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=STAGE2_EPOCHS,

    callbacks=callbacks_stage2

)



print("\n🎉 Complete training finished.")



# Save final Keras model

model.save(
    "/content/age_prediction_final.keras"
)


print(
    "💾 Saved: /content/age_prediction_final.keras"
)



🚀 Starting Stage-1 Transfer Learning Training...
Epoch 1/5
593/593 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - classification_output_accuracy: 0.8570 - classification_output_loss: 0.4543 - loss: 1.3292 - regression_output_loss: 17.4975 - regression_output_mae: 17.9893 

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()



Epoch 1: saving model to /content/drive/MyDrive/Appblocker/Model_Checkpoints/age_model_stage1_epoch_01.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Appblocker/Model_Checkpoints/age_model_stage1_epoch_01.keras
593/593 ━━━━━━━━━━━━━━━━━━━━ 7712s 13s/step - classification_output_accuracy: 0.8734 - classification_output_loss: 0.4019 - loss: 1.0569 - regression_output_loss: 13.0995 - regression_output_mae: 13.5909 - val_classification_output_accuracy: 0.9059 - val_classification_output_loss: 0.2922 - val_loss: 0.7825 - val_regression_output_loss: 9.7897 - val_regression_output_mae: 10.2882 - learning_rate: 0.0010
Epoch 2/5
593/593 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - classification_output_accuracy: 0.8914 - classification_output_loss: 0.3408 - loss: 0.8436 - regression_output_loss: 10.0562 - regression_output_mae: 10.5430
Epoch 2: saving model to /content/drive/MyDrive/Appblocker/Model_Checkpoints/age_model_stage1_epoch_02.keras

Epoch 2: finished saving model to /content/d

## 7. Model Evaluation

Now that the training is complete, let's evaluate the model on the validation dataset to understand its performance.

In [ ]:
print("\n✨ Evaluating the model on the validation set...")

# Evaluate the model
results = model.evaluate(
    val_generator,
    verbose=1
)

# Map results to their corresponding metric names
# The order of metrics in `model.evaluate` output corresponds to `model.metrics_names`
metrics_names = model.metrics_names

# Create a dictionary for easier access to results
evaluation_results = dict(zip(metrics_names, results))

print("\n--- Model Evaluation Results ---")
for metric, value in evaluation_results.items():
    print(f"  {metric}: {value:.4f}")

print("\n✅ Evaluation complete!")



✨ Evaluating the model on the validation set...
149/149 ━━━━━━━━━━━━━━━━━━━━ 202s 1s/step - classification_output_accuracy: 0.9372 - classification_output_loss: 0.1959 - loss: 0.5106 - regression_output_loss: 6.2654 - regression_output_mae: 6.7591

--- Model Evaluation Results ---
  loss: 0.5106
  compile_metrics: 0.1959
  classification_output_loss: 6.2654
  regression_output_loss: 0.9372

✅ Evaluation complete!


### Detailed Performance Analysis (Optional)

To get a more detailed view of the classification and regression performance, we can extract predictions and compute metrics like a classification report (precision, recall, f1-score) and regression error metrics (like MAE or RMSE) separately.

In [ ]:
from sklearn.metrics import classification_report, mean_absolute_error

print("\n📊 Generating detailed performance metrics...")

# Make predictions on the validation set
predictions = model.predict(val_generator)

# Extract classification and regression predictions
cls_preds = predictions[0] # First output is classification
reg_preds = predictions[1] # Second output is regression

# Extract true labels from the validation generator
true_cls_labels = []
true_reg_labels = []

for i in range(len(val_generator)):
    _, labels = val_generator[i]
    true_cls_labels.extend(tf.argmax(labels['classification_output'], axis=1).numpy())
    # Fix: Remove redundant .numpy() call as labels['regression_output'] is already a numpy array
    true_reg_labels.extend(labels['regression_output'].flatten())

# Convert predictions to class labels for classification report
predicted_cls_labels = tf.argmax(cls_preds, axis=1).numpy()

# --- Classification Report ---
# Assuming age_classes: 0: Kid (<13), 1: Teen (13-19), 2: Adult (>19)
class_names = ['Kid', 'Teen', 'Adult']
print("\n--- Classification Report ---")
print(classification_report(true_cls_labels, predicted_cls_labels, target_names=class_names))

# --- Regression Metrics ---
print("\n--- Regression Metrics ---")
mae = mean_absolute_error(true_reg_labels, reg_preds)
print(f"Mean Absolute Error (MAE): {mae:.4f} years")

# Optional: You can also calculate RMSE
rmse = np.sqrt(tf.keras.metrics.MeanSquaredError()(true_reg_labels, reg_preds).numpy())
print(f"Root Mean Squared Error (RMSE): {rmse:.4f} years")

print("\nDetailed analysis complete.")



📊 Generating detailed performance metrics...
149/149 ━━━━━━━━━━━━━━━━━━━━ 193s 1s/step

--- Classification Report ---
              precision    recall  f1-score   support

         Kid       0.90      0.85      0.88       675
        Teen       0.54      0.29      0.38       211
       Adult       0.95      0.99      0.97      3856

    accuracy                           0.94      4742
   macro avg       0.80      0.71      0.74      4742
weighted avg       0.93      0.94      0.93      4742


--- Regression Metrics ---
Mean Absolute Error (MAE): 6.7591 years
Root Mean Squared Error (RMSE): 9.7029 years

Detailed analysis complete.


In [ ]:
import os
import tensorflow as tf
import numpy as np # Added for np.float32 type casting
import cv2 # Required for preprocess_inplace_image
import gc # Required for RAMProofLazyGenerator (if gc.collect() was used)
from sklearn.model_selection import train_test_split # Required for data splitting
import zipfile # Required for unzipping

# Define CHECKPOINT_DIR here to ensure it's always available
CHECKPOINT_DIR = "/content/drive/MyDrive/Appblocker/Model_Checkpoints"

# --- Dependencies for RAMProofLazyGenerator and val_generator to make this cell self-contained ---
# Define global parameters used by the generator and preprocessor (from YzA9bCFJJ9ZQ)
IMG_SIZE = 224
BATCH_SIZE = 32

# Data paths (from YzA9bCFJJ9ZQ)
ZIP_FILE_PATH = "/content/drive/MyDrive/Appblocker/UTKFace.zip"
EXTRACTED_DIR_PATH = "/content/drive/MyDrive/Appblocker/UTKFace_extracted"

# Ensure the checkpoint directory exists (already done in a prior cell, but good for robustness)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Unzip the file if the extracted directory does not exist (from YzA9bCFJJ9ZQ)
if not os.path.exists(EXTRACTED_DIR_PATH):
    print(f"Unzipping '{ZIP_FILE_PATH}' to '{EXTRACTED_DIR_PATH}'...")
    with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACTED_DIR_PATH)
    print("Unzipping complete.")
else:
    print(f"Extracted directory '{EXTRACTED_DIR_PATH}' already exists. Skipping unzipping.")

# Adjust EXTRACTED_DIR_PATH to point to the specific 'UTKFace' subdirectory (from YzA9bCFJJ9ZQ)
UTKFACE_SUBDIR_PATH = os.path.join(EXTRACTED_DIR_PATH, 'UTKFace')
if os.path.exists(UTKFACE_SUBDIR_PATH):
    EXTRACTED_DIR_PATH = UTKFACE_SUBDIR_PATH
else:
    print(f"Warning: '{UTKFACE_SUBDIR_PATH}' not found. Using '{EXTRACTED_DIR_PATH}' as fallback.")

# Fetch every available image filename (from YzA9bCFJJ9ZQ)
all_files_full_paths = []
for root, dirs, files in os.walk(EXTRACTED_DIR_PATH):
    for file in files:
        if file.endswith('.jpg'):
            all_files_full_paths.append(os.path.join(root, file))
all_files_full_paths.sort()

# Download Haar Cascade file if not present (from QcJslk_ELNbk) - CRITICAL FOR FIX
cascade_path = "/content/haarcascade_frontalface_default.xml"
if not os.path.exists(cascade_path):
    print("Downloading Haar Cascade file...")
    !wget -O {cascade_path} https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml
    print("Haar Cascade file downloaded.")

# Load Haar Cascade (from QcJslk_ELNbk and YzA9bCFJJ9ZQ)
face_cascade = cv2.CascadeClassifier(cascade_path)
if face_cascade.empty():
    # This should ideally not happen after the wget, but good for robust error checking
    raise RuntimeError("❌ Haar Cascade failed to load after download attempt.")

clahe = cv2.createCLAHE( # From QcJslk_ELNbk
    clipLimit=2.0,
    tileGridSize=(8, 8)
)

def preprocess_inplace_image(image_path, out_matrix, index): # From QcJslk_ELNbk
    """Processes image and injects it directly into an allocated memory array slot."""
    img = cv2.imread(image_path)

    if img is None:
        out_matrix[index] = np.zeros(
            (IMG_SIZE, IMG_SIZE, 3),
            dtype=np.float32
        )
        return

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5
    )

    if len(faces) == 0:
        h, w = img.shape[:2]
        sx = int(w * 0.10)
        sy = int(h * 0.10)
        ex = int(w * 0.90)
        ey = int(h * 0.90)
        face = img[sy:ey, sx:ex]
    else:
        x, y, w, h = max(
            faces,
            key=lambda f: f[2] * f[3]
        )
        face = img[y:y+h, x:x+w]

    ycrcb = cv2.cvtColor(
        face,
        cv2.COLOR_BGR2YCrCb
    )
    channels = list(cv2.split(ycrcb))
    channels[0] = clahe.apply(channels[0])
    ycrcb = cv2.merge(channels)
    rgb = cv2.cvtColor(
        ycrcb,
        cv2.COLOR_YCrCb2RGB
    )
    rgb = cv2.resize(
        rgb,
        (IMG_SIZE, IMG_SIZE)
    )
    rgb = rgb.astype(np.float32)
    # Assuming MobilenetV3 preprocess_input is desired here as per cell QcJslk_ELNbk
    rgb = tf.keras.applications.mobilenet_v3.preprocess_input(rgb)
    out_matrix[index] = rgb

class RAMProofLazyGenerator(tf.keras.utils.Sequence): # From bYMZNVKyLQQM
    """
    Efficient Keras data generator that loads images batch-by-batch,
    preprocesses them in-place, and supports shuffling.
    """

    def __init__(
        self,
        paths,
        ages,
        classes,
        batch_size=32,
        shuffle=True
    ):
        self.paths = np.array(paths)
        self.ages = np.array(ages, dtype=np.float32)
        self.classes = np.array(classes, dtype=np.int32)

        self.batch_size = batch_size
        self.shuffle = shuffle

        self.batch_images_buffer = np.zeros(
            (batch_size, IMG_SIZE, IMG_SIZE, 3),
            dtype=np.float32
        )

        self.indices = np.arange(len(self.paths))

        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.paths) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):

        start = idx * self.batch_size
        end = min((idx + 1) * self.batch_size, len(self.paths))

        batch_indices = self.indices[start:end]

        current_batch_size = len(batch_indices)

        for i, sample_index in enumerate(batch_indices):

            preprocess_inplace_image(
                self.paths[sample_index],
                self.batch_images_buffer,
                i
            )

        batch_age = self.ages[batch_indices]

        batch_cls = tf.keras.utils.to_categorical(
            self.classes[batch_indices],
            num_classes=3
        )

        # Removed gc.collect() as it's detrimental to performance and multiprocessing should handle memory better
        # Note: Original RAMProofLazyGenerator from cell bYMZNVKyLQQM did not have gc.collect()
        # The version in YzA9bCFJJ9ZQ had it conditionally, but later removed.
        # Sticking to the version without it from bYMZNVKyLQQM.

        return (
            self.batch_images_buffer[:current_batch_size],
            {
                "classification_output": batch_cls,
                "regression_output": batch_age,
            },
        )

# Parse raw text and split data (from YzA9bCFJJ9ZQ)
file_paths = []
ages = []
age_classes = [] # 0: Kid (<13), 1: Teen (13-19), 2: Adult (>19)

for full_path in all_files_full_paths:
    filename = os.path.basename(full_path)
    parts = filename.split('_')
    if len(parts) >= 3:
        try:
            age = int(parts[0])
            if age < 13:
                cls = 0
            elif age <= 19:
                cls = 1
            else:
                cls = 2

            ages.append(age)
            age_classes.append(cls)
            file_paths.append(full_path)
        except ValueError:
            continue

file_paths = np.array(file_paths)
ages = np.array(ages, dtype=np.float32)
age_classes = np.array(age_classes, dtype=np.int32)

# Split arrays into Training and Validation (from YzA9bCFJJ9ZQ)
tr_paths, val_paths, tr_ages, val_ages, tr_cls, val_cls = train_test_split(
    file_paths, ages, age_classes, test_size=0.2, random_state=42
)
# --- End Dependencies for RAMProofLazyGenerator ---


print("📦 Initiating Post-Training Quantization into compressed INT8 Engine for Android...")

# 1. Load the finalized peak trained Keras model structure
# Changed to load from Google Drive checkpoint as /content/ is ephemeral.
# Based on training logs, the best epoch was 14 for Stage 2.
final_model_path = os.path.join(CHECKPOINT_DIR, "age_model_final_epoch_14.keras")

if not os.path.exists(final_model_path):
    raise FileNotFoundError(f"🚨 ERROR: Cannot find the model at {final_model_path}. Ensure training completed successfully and check the correct epoch number if the best epoch changed.")

model = tf.keras.models.load_model(final_model_path)
print(f"✅ Final Keras model successfully loaded from Drive: '{final_model_path}' into conversion pipeline.")

# 2. Convert the standard Keras structural model layer graphs to TFLite format
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Instantiate val_generator here to ensure its availability for the representative_dataset.
val_generator = RAMProofLazyGenerator(
    val_paths,
    val_ages,
    val_cls,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# 3. Representative calibration function feeding your unique MobileNet-preprocessed floats
# Using a class to explicitly capture val_generator for TFLiteConverter's scope.
class RepresentativeDataset:
    def __init__(self, generator):
        self.generator = generator

    def __call__(self):
        # Pass 30 mini-batches from the validation data stream to evaluate range variations
        for i in range(30):
            # Ensure that self.generator[i] returns the expected tuple (img_batch, labels)
            img_batch, _ = self.generator[i]
            yield [img_batch[0:1].astype(np.float32)]

converter.representative_dataset = RepresentativeDataset(val_generator)

# 4. Bind operational hardware components strictly to 8-bit integer ranges
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.float32
converter.inference_output_type = tf.float32

quantized_tflite_model = converter.convert()

# 5. Export final binary files across your Google Drive partitions and local cloud workspace
tflite_drive_path = "/content/drive/MyDrive/Appblocker/age_model_quantized.tflite"
with open(tflite_drive_path, "wb") as f:
    f.write(quantized_tflite_model)

with open("/content/age_model_quantized.tflite", "wb") as f:
    f.write(quantized_tflite_model)

print("\n🎉 CONVERSION COMPLETE!")
print(f"👉 File saved to your Drive: '{tflite_drive_path}'")

Extracted directory '/content/drive/MyDrive/Appblocker/UTKFace_extracted' already exists. Skipping unzipping.
--2026-08-09 08:02:39--  https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 930127 (908K) [text/plain]
Saving to: ‘/content/haarcascade_frontalface_default.xml’

/content/haarcascad 100%[===================>] 908.33K  --.-KB/s    in 0.006s  

2026-08-09 08:02:39 (144 MB/s) - ‘/content/haarcascade_frontalface_default.xml’ saved [930127/930127]

Haar Cascade file downloaded.
📦 Initiating Post-Training Quantization into compressed INT8 Engine for Android...
✅ Final Keras model successfully loaded from Drive: '/content/drive/MyDrive/Appblocker/Model_

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



🎉 CONVERSION COMPLETE!
👉 File saved to your Drive: '/content/drive/MyDrive/Appblocker/age_model_quantized.tflite'
